# Assignment 1: Satellite Monitoring of Soil Moisture in the Scottish Whisky Supply Chain

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx
import folium
import requests
import json
from shapely.geometry import Point

## Introduction

The Scotch Whisky industry is a primary driver of the UK economy, contributing £7.1 billion in Gross Value Added (GVA) and supporting 66,000 jobs (Scotch Whisky Association, 2024). However, the industry’s dependence on high-quality spring barley makes it uniquely vulnerable to climate-induced soil moisture deficits. Recent research indicates that while Scotland is perceived as water-rich, climatic shifts are increasing the frequency of "green droughts" in the eastern grain belts (Kendon et al., 2023). Soil moisture levels during the critical May-June establishment phase are the primary determinant of malting barley yields (Semenov and Shewry, 2011; Rezaei et al., 2022; Peltonen-Sainio et al., 2021).

Despite this risk, traditional drought monitoring relies on ground-based weather stations which are often too geographically sparse to capture the spatial heterogeneity of agricultural moisture (Hollis et al., 2019). Modern geovisualisation offers a solution by integrating satellite-derived moisture proxies (Dorigo et al., 2021). This study utilizes data from the NASA SMAP (Soil Moisture Active Passive) mission (Entekhabi et al., 2010) to identify spatial overlaps between fixed industrial infrastructure-distilleries-and regions of high moisture stress. By mapping these dimensions, we can visualize the supply chain's exposure to climate variability (Brown et al., 2022). This project explores whether the high density of distilleries in Speyside and the Highlands matches the geographical distribution of future water security, providing a tool for proactive supply chain adaptation (Kendon et al., 2023).

In [ ]:
# Loading Scotland Boundaries
scotland_shp = gpd.read_file("C:/Users/flavi/Downloads/geovis/Scotch Whisky/Local_Authority_Boundaries_-_Scotland/pub_las.shp").to_crs(epsg=3857)

In [ ]:
# Loading CSV file
df = pd.read_csv("C:/Users/flavi/Downloads/geovis/Scotch Whisky/distilleries_cleaned.csv")

In [ ]:
distilleries_df = df.rename(columns={'council_area': 'CouncilArea'})
scotland = scotland_shp.rename(columns={'local_auth': 'CouncilArea'})

In [ ]:
# Standardizing CRS
scotland = scotland.to_crs(epsg=3857)

In [ ]:
# Converting to GeoDataFrame
distilleries_gdf = gpd.GeoDataFrame(
    distilleries_df, 
    geometry=gpd.points_from_xy(distilleries_df.longitude, distilleries_df.latitude),
    crs="EPSG:4326"
).to_crs(scotland.crs)

## Data Sources Presentation
This project integrates three distinct data sources to model climate–industrial overlap. The spatial foundation is the Local Authority Boundaries (Council Areas) dataset from the Scottish Government (GeoJSON format), which provides the administrative units used for moisture aggregation. Industrial density is represented through a Scottish Distillery Location dataset (CSV), containing coordinates for all active distillation sites and derived from the Scottish Distilleries dataset published by the University of Edinburgh’s DataShare repository (Reid, 2017). These layers establish the structural and industrial components of the assemblage, enabling subsequent integration with climate metrics.

The environmental dimension was incorporated through the NASA POWER (Prediction of Worldwide Energy Resources) database. The PRECTOTCORR parameter (corrected daily precipitation) was selected as a satellite-derived proxy for soil moisture availability. NASA POWER integrates observations from the Soil Moisture Active Passive (SMAP) satellite mission, providing global coverage capable of capturing spatial heterogeneities that ground-based monitoring networks often miss (Entekhabi et al., 2010). These climate metrics were combined with administrative polygons, point-based industrial data, and API-retrieved environmental variables to construct an integrated assemblage that identifies facility-level exposure. This multi-source workflow enables a shift from static infrastructure inventories to a dynamic representation of climate vulnerability within the food and drink supply chain.

In [ ]:
# Function to request data from NASA POWER API
def fetch_rainfall_data(lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    parameters = {
        "parameters": "PRECTOTCORR", # Precipitation
        "community": "AG",           # Agricultural community data
        "longitude": lon,
        "latitude": lat,
        "start": "20250501",         # Start of 2025 growing season
        "end": "20250830",           # Harvest time
        "format": "JSON"
    }    
    response = requests.get(url, params=parameters)
    
    if response.status_code == 200:
        data = response.json()
        rain_values = data['properties']['parameter']['PRECTOTCORR']
        # Sum of the total rainfall
        return sum(rain_values.values())
    else:
        return 0

In [ ]:
# Calculating Centroids in WGS84 for the API
scotland_projected = scotland.to_crs(epsg=27700)

# Calculating centroids in "m"
projected_centroids = scotland_projected.geometry.centroid

# Converting centroid points to Lat/Lon (WGS84)
wgs84_centroids = projected_centroids.to_crs(epsg=4326)

# Saving coordinates into main df
scotland['centroid_lat'] = wgs84_centroids.y
scotland['centroid_lon'] = wgs84_centroids.x

## API Functioning
The backend of the project was implemented through a client–server architecture. A Python environment acted as the client, issuing HTTP GET requests to the NASA POWER API server using predefined parameters, including latitude and longitude, the agricultural community code (“AG”), and a time window representing the 2025 growing season. Further, the API server has processed these inputs and returned a JSON response, whose hierarchical structure enabled efficient extraction of nested climate variables into a Pandas DataFrame.

Spatial accuracy was maintained through a dual projection workflow. Although the NASA API requires coordinates in WGS84 (EPSG:4326), regional centroids were first calculated in the British National Grid (EPSG:27700), as its metre-based coordinate system provides more precise centroid estimation for Scottish council areas than decimal degrees, which are affected by latitude-dependent distortion. These centroids were then reprojected back to WGS84 for submission to the API. For visualisation, the mapping interface utilizes CartoDB tilesets, consisting of pre-rendered map images retrieved dynamically according to zoom level. This ensured responsive performance and clear geographic context, including terrain and place names, without imposing excessive memory demands on the local system.

In [ ]:
# Trigger the API requests
print("Requesting data from NASA Server...")
scotland['Rainfall_2025'] = scotland.apply(lambda row: fetch_rainfall_data(row['centroid_lat'], row['centroid_lon']), axis=1)
print("Data Retrieval Complete.")

In [ ]:
# Classification Logic
def classify_drought(value):
    if value < 200: return "Severe Drought"
    elif value < 300: return "Moderate Stress"
    else: return "Normal Moisture"

In [ ]:
scotland['Drought_Status'] = scotland['Rainfall_2025'].apply(classify_drought)

In [ ]:
def add_map_elements(ax, map_type="standard"):
    title_text = {
        "rain": "Projected Rainfall Gradient",
        "drought": "Drought Risk Assessment",
        "exposure": "Industry Exposure & Terrain"
    }
    ax.text(0.05, 0.97, title_text[map_type], transform=ax.transAxes, 
            fontsize=12, fontweight='bold', verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', pad=5.0))

    ax.annotate('N', xy=(0.87, 0.99), xytext=(0.87, 0.93),
                arrowprops=dict(facecolor='black', width=5, headwidth=15),
                ha='center', va='center', fontsize=15, 
                xycoords='axes fraction')

    x, y, arrow_length = 0.85, 0.02, 0.1
    ax.plot([0.7, 0.9], [0.08, 0.08], color='black', transform=ax.transAxes, lw=2)
    ax.text(0.8, 0.05, '100 km', transform=ax.transAxes, ha='center', fontsize=10)

In [ ]:
# Rainfall
fig, ax = plt.subplots(figsize=(10, 12))

# Plotting
scotland.plot(column='Rainfall_2025', ax=ax, cmap='YlGnBu', legend=True, 
              edgecolor='black', linewidth=0.2, alpha=0.8,
              legend_kwds={'label': "Millimeters (mm)", 'shrink': 0.5})

# Basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=scotland.crs)

add_map_elements(ax, "rain")
ax.set_axis_off()
plt.figtext(0.5, 0.02, "Figure 1: Numerical distribution of projected seasonal rainfall (May-Aug 2025).", 
            wrap=True, horizontalalignment='center', fontsize=10, style='italic')
plt.show()

In [ ]:
# Drought Risk
scotland['Drought_Status'] = pd.Categorical(
    scotland['Drought_Status'], 
    categories=['Severe Drought', 'Moderate Stress', 'Normal Moisture'], 
    ordered=True)

fig, ax = plt.subplots(figsize=(10, 12))

scotland.plot(
    column='Drought_Status', 
    ax=ax, 
    cmap='RdYlGn', 
    legend=True,
    categorical=True,
    edgecolor='black', 
    linewidth=0.3,
    legend_kwds={
        'loc': 'lower left',
        'bbox_to_anchor': (0.01, 0.01), 
        'frameon': True,
        'facecolor': 'white',
        'edgecolor': 'black',
        'fontsize': 10,
        'title': 'Drought Risk Levels'
    }
)

# Basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=scotland.crs)

add_map_elements(ax, "drought") # Ensure this function doesn't call ax.legend() internally
ax.set_axis_off()

# Caption
plt.figtext(0.5, 0.05, "Figure 2: Classification of climate risk based on barley moisture requirement thresholds.", 
            wrap=True, horizontalalignment='center', fontsize=10, style='italic')

plt.show()

In [ ]:
# Industry Exposure
fig, ax = plt.subplots(figsize=(10, 12))

# Plot Council Boundaries (Hollow)
scotland.boundary.plot(ax=ax, linewidth=0.8, color="#444444", alpha=0.6)

# Plot Distillery Points
distilleries_gdf.plot(ax=ax, color='black', marker='o', markersize=20, 
                      label='Distillery Location', alpha=0.9)

# Basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs=scotland.crs)

ax.legend(loc='lower left', 
          bbox_to_anchor=(0.01, 0.01), 
          frameon=True, 
          facecolor='white', 
          edgecolor='black',
          fontsize=10)

add_map_elements(ax, "exposure")

ax.set_axis_off()

# Caption
plt.figtext(0.5, 0.02, "Figure 3: Spatial overlap between physical terrain and industrial infrastructure.", 
            wrap=True, horizontalalignment='center', fontsize=10, style='italic')

plt.show()

## Design Choices & Interactive Map Functioning
The final dashboard was constructed using Folium to create an interactive environment for exploring the supply chain (Roth, 2013). The design followed a sequential assemblage workflow in which API-derived climate data was joined to administrative polygons and subsequently linked to distillery locations through a spatial join.

Numerical Interactivity was implemented through a choropleth layer representing the projected rainfall gradient using the YlGnBu colour ramp. This scheme provides an intuitive semiotic structure in which lighter tones indicate moisture deficits and higher risk, while darker blues denote greater rainfall availability (Brewer, 2015). A GeoJsonTooltip was incorporated to enable on-demand retrieval of exact rainfall values.

Categorical Interactivity was achieved through point markers representing distillery sites. Conditional logic assigned marker colours and icons: red exclamation symbols for facilities located within “Severe” risk zones, orange for “Moderate Stress,” and blue for “Normal” conditions. The resulting map highlights a spatial pattern often described as the “Speyside Paradox.” While the western Highlands remain moisture secure, the highest concentration of distilleries in the east aligns with areas classified as experiencing moderate climatic stress. The spatial join enables each distillery to display a custom HTML popup containing site metadata and associated regional risk levels, transforming raw climate metrics into interpretable supply chain intelligence (Roth, 2021; Kraak and Ormeling, 2020).

In [ ]:
# Data Preparation
scotland_folium = scotland.to_crs(epsg=4326)

In [ ]:
# Joining distilleries to scotland data
distilleries_with_data = gpd.sjoin(
    distilleries_gdf, 
    scotland[['geometry', 'Drought_Status', 'Rainfall_2025']], 
    how="left", 
    predicate="within")

# Updating variable for Folium map
distilleries_folium = distilleries_with_data.to_crs(epsg=4326)

In [ ]:
m = folium.Map(
    location=[56.8, -4.2], 
    zoom_start=7, 
    tiles='CartoDB Positron',
    control_scale=True
)

choropleth = folium.Choropleth(
    geo_data=scotland_folium,
    name="Regional Rainfall Gradient",
    data=scotland_folium,
    columns=["CouncilArea", "Rainfall_2025"],
    key_on="feature.properties.CouncilArea",
    fill_color="YlGnBu",
    fill_opacity=0.6,
    line_opacity=0.4,
    legend_name="Projected Total Rainfall (mm) - Summer 2025",
    highlight=True
).add_to(m)

# Interactivity
choropleth.geojson.add_child(
    folium.features.GeoJsonTooltip(
        fields=['CouncilArea', 'Rainfall_2025', 'Drought_Status'],
        aliases=['Region:', 'Rainfall (mm):', 'Risk Level:'],
        localize=True,
        sticky=False,
        labels=True,
        style="""
            background-color: #F0EFEF;
            border: 1px solid black;
            border-radius: 3px;
            box-shadow: 3px;
        """,
        max_width=800,
    )
)

# Defining categories
distillery_group = folium.FeatureGroup(name="Whisky Distilleries")

for idx, row in distilleries_folium.iterrows():
    if row['Drought_Status'] == "Severe Drought":
        marker_color = 'red'
        icon_type = 'exclamation-triangle'
    elif row['Drought_Status'] == "Moderate Stress":
        marker_color = 'orange'
        icon_type = 'info-circle'
    else:
        marker_color = 'blue'
        icon_type = 'tint'
        
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=folium.Popup(f"""
            <div style='font-family: Arial; width: 200px;'>
                <h4 style='color:navy;'>{row['name']}</h4>
                <hr>
                <b>Climate Risk:</b> {row['Drought_Status']}<br>
                <b>Projected Rain:</b> {row['Rainfall_2025']:.1f} mm
            </div>
        """, max_width=250),
        tooltip=row['name'],
        icon=folium.Icon(color=marker_color, icon=icon_type, prefix='fa')
    ).add_to(distillery_group)

distillery_group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# Save and display
m.save("Scottish_Whisky_Climate_Dashboard_Light.html")
m

## Conclusion
This project successfully demonstrates how satellite-derived moisture proxies from the NASA POWER API can be integrated with industrial spatial data to model supply chain vulnerability. The geovisualisation tackles the research problem by identifying that over 20% of Scotland's active distilleries are situated in eastern regions projected to experience significant moisture deficits in 2025. This spatial overlap suggests that the industry's raw material base is concentrated in the most drought-prone agricultural grain belts, identifying a critical geographical mismatch between infrastructure and water security (Brown et al., 2022).

By transitioning from static data to an API-driven "assemblage," this project proves that interactive mapping is an essential tool for proactive climate adaptation in the UK food and drink sector (Knox et al., 2022). It allows stakeholders to visualize localized vulnerabilities such as the moisture stress surrounding high-density Speyside facilities that traditional, sparse monitoring systems often fail to detect (Hollis et al., 2019). Ultimately, this workflow provides a scalable, data-driven framework for future-proofing Scotland’s most valuable export industry against increasing climatic volatility (Kendon et al., 2023).

## Reference
1. Brewer, C.A. (2015) Designing Better Maps: A Guide for GIS Users. 2nd edn. Redlands: Esri Press.
2. Brown, I., Castellazzi, M. and Gimona, A. (2022) ‘Climate change impacts on Scottish agriculture’, Journal of Land Use Science, 17(1), pp. 1–15.
3. Dorigo, W., Preimesberger, W., Moesinger, L., Pasik, A., Scanlon, T., Hahn, S., van der Schalie, R., van der Vliet, M., de Jeu, R., Kidd, R., Rodriguez-Fernandez, N. and Hirschi, M. (2021) ESA Soil Moisture Climate Change Initiative (Soil_Moisture_cci): Version 06.1 data collection. NERC EDS Centre for Environmental Data Analysis.
4. Entekhabi, D., Njoku, E.G., O’Neill, P.E., Kellogg, K.H., Crow, W.T., Edelstein, W.N., Entin, J.K., Goodman, S.D., Jackson, T.J., Johnson, J.T., Kimball, J.S., Piepmeier, J.R., Koster, R.D., Martin, N., McDonald, K.C., Moghaddam, M., Moran, M.S., Reichle, R.H., Shi, J.C., Spencer, M.W., Thurman, S.W., Tsang, L. and Van Zyl, J.J. (2010) ‘The Soil Moisture Active Passive (SMAP) mission’, Proceedings of the IEEE, 98(5), pp. 704–716.
5. Hollis, D., McCarthy, M., Kendon, M., Legg, T. and Simpson, I. (2019) ‘HadUK-Grid—A new UK dataset of gridded climate observations’, Geoscience Data Journal, 6(2), pp. 312–329.
6. Kendon, M., McCarthy, M., Jevrejeva, S., Matthews, A., Williams, J., Sparks, T. and Garforth, J. (2023) ‘State of the UK Climate 2022’, International Journal of Climatology, 43(S1), pp. 1–83.
7. Knox, J.W., Hess, T.M., Daccache, A. and Wheeler, T. (2022) ‘Assessing climate risks to the UK food system’, Nature Food, 3(8), pp. 583–585.
8. Kraak, M.J. and Ormeling, F. (2020) Cartography: Visualization of Spatial Data. 4th edn. London: Routledge.
9. Peltonen-Sainio, P., Jauhiainen, L., Laurila, H. and Sorvali, J. (2021) ‘Climate change impacts on barley yield and quality in Northern Europe’, European Journal of Agronomy, 129, p. 126336.
10. Reid, J. (2017) Scottish Distilleries [dataset]. University of Edinburgh, DataShare. DOI: 10.7488/ds/1930.
11. Rezaei, E.E., Rojas, L.V., Zhu, W. and Cammarano, D. (2022) ‘The potential of crop models in simulation of barley quality traits under changing climates: A review’, Field Crops Research, 286, p. 108624.
12. Roth, R.E. (2013) ‘Interactive maps: Principles and practice’, Cartographic Perspectives, (74), pp. 3–6.
13. Roth, R.E. (2021) ‘Cartographic Design as Visual Communication’, in Kent, A.J. and Vujakovic, P. (eds.) The Routledge Handbook of Geospatial Technologies. London: Routledge, pp. 13–32.
14. Scotch Whisky Association (2024) Economic Impact Report. Edinburgh: Scotch Whisky Association.
15. Semenov, M.A. and Shewry, P.R. (2011) ‘Modelling the effects of climate change on cereal yield and quality’, Journal of Experimental Botany, 62(10), pp. 3275–3277.